# EDA Terreno: limpieza y estandarización

Notebook para limpiar `terrenos_idealista.csv` con foco en nulos, tipado, estandarización de `tipo_suelo` y variables derivadas.


In [37]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd


## Carga del dataset

Se lee el CSV desde el repositorio para trabajar siempre con la versión local de esta rama.


In [38]:
csv_path = Path("..") / "data" / "raw" / "scaping_manual" / "terrenos_idealista.csv"
df = pd.read_csv(csv_path, sep=";")

print(f"Filas: {len(df):,} | Columnas: {df.shape[1]}")
df.head(3)


Filas: 1,080 | Columnas: 12


,id,anunciante,titulo,municipio,zona,precio_eur,precio_anterior_eur,descuento_pct,superficie_m2,tipo_suelo,referencia_catastral,notas
0,1,Exclusivas inmobiliarias mikeli,"Terreno en Barrio Llanilla, Alisal - Cazoña - ...",Santander,"Barrio Llanilla, Alisal - Cazoña - San Román",430000,NaN,NaN,972,NaN,NaN,NaN
1,2,MENA INMOBILIARIA,"Terreno en Oruña, Piélagos",Piélagos,Oruña,139000,NaN,NaN,965,Urbano (solar),NaN,Edificabilidad 200 m²
2,3,Grupo Inmobiliario Tribeca,"Terreno en Peñacastillo - Nuevamontaña, Santander",Santander,Peñacastillo - Nuevamontaña,169000,NaN,NaN,650,Urbano (solar),NaN,5 parcelas


## Normalización inicial y tratamiento de nulos

1. Se limpia texto (espacios).
2. Se convierten marcadores vacíos a nulo real (`NaN`).
3. Se identifican registros con al menos una variable nula.


In [39]:
# Estandarizar vacíos y marcadores de no dato en todo el dataframe.
missing_tokens = {
    "", " ", "na", "n/a", "none", "null", "nan", "-", "--", "sin dato", "no informado"
}

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({token: np.nan for token in missing_tokens})

# Máscara de registros incompletos (alguna variable nula).
mask_registro_incompleto = df.isna().any(axis=1)

print("Registros con al menos un nulo:", int(mask_registro_incompleto.sum()))
print("Porcentaje sobre total:", round(mask_registro_incompleto.mean() * 100, 2), "%")

df.loc[mask_registro_incompleto].head(10)


Registros con al menos un nulo: 1079
Porcentaje sobre total: 99.91 %


,id,anunciante,titulo,municipio,zona,precio_eur,precio_anterior_eur,descuento_pct,superficie_m2,tipo_suelo,referencia_catastral,notas
0,1,Exclusivas inmobiliarias mikeli,"Terreno en Barrio Llanilla, Alisal - Cazoña - ...",Santander,"Barrio Llanilla, Alisal - Cazoña - San Román",430000,NaN,NaN,972,NaN,NaN,NaN
1,2,MENA INMOBILIARIA,"Terreno en Oruña, Piélagos",Piélagos,Oruña,139000,NaN,NaN,965,Urbano (solar),NaN,Edificabilidad 200 m²
2,3,Grupo Inmobiliario Tribeca,"Terreno en Peñacastillo - Nuevamontaña, Santander",Santander,Peñacastillo - Nuevamontaña,169000,NaN,NaN,650,Urbano (solar),NaN,5 parcelas
3,4,Exclusivas inmobiliarias mikeli,"Terreno en Calle de la Arnia, Soto de la Marin...",Santa Cruz de Bezana,"Calle de la Arnia, Soto de la Marina",1290000,NaN,NaN,3217,NaN,NaN,Parcela urbana
4,5,Exclusivas inmobiliarias mikeli,"Terreno en Calle Barrio San Roque, Polanco",Polanco,Calle Barrio San Roque,80000,NaN,NaN,1668,NaN,NaN,820 urbanos + 848 rústicos
5,6,Exclusivas inmobiliarias mikeli,"Terreno en Avenida Playa San Juan de la Canal,...",Santa Cruz de Bezana,"Av. Playa San Juan de la Canal, Soto de la Marina",1500000,NaN,NaN,6518,NaN,NaN,3.433 urbanos
6,7,Bahía Home,"Terreno en Barrio la Venera, Oruña, Piélagos",Piélagos,"Barrio la Venera, Oruña",98000,NaN,NaN,4507,Urbanizable,NaN,Licencia + proyecto
7,8,Distrito inmobiliario,"Terreno en Barrio Mar, 87 --81, Polanco",Polanco,"Barrio Mar, 87 --81",132000,140000.0,6.0,2210,Urbano (solar),NaN,Licencia + proyecto básico
8,9,RE/MAX Vértice,"Terreno en Sv-4713, 12, Miengo",Miengo,"Sv-4713, 12",300000,NaN,NaN,6548,No urbanizable,NaN,Proyecto hotel canino
9,10,RE/MAX Vértice,"Terreno en Barrio la Picota, 38 a, Renedo, Pié...",Piélagos,"Barrio la Picota, 38 a, Renedo",69000,NaN,NaN,413,Urbano (solar),NaN,NaN


## Tipado y saneamiento de variables numéricas

Se fuerzan a numérico las variables cuantitativas; cualquier texto o formato inválido se transforma en nulo.


In [40]:
def to_numeric_strict(series: pd.Series) -> pd.Series:
    # Convierte texto a número; si hay texto no numérico, deja NaN.
    cleaned = (
        series.astype(str)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan, "None": np.nan})
        .str.replace(r"[^0-9,.-]", "", regex=True)
        .str.replace(",", ".", regex=False)
    )
    return pd.to_numeric(cleaned, errors="coerce")

numeric_cols = ["id", "precio_eur", "precio_anterior_eur", "descuento_pct", "superficie_m2"]
for col in numeric_cols:
    df[col] = to_numeric_strict(df[col])

df["id"] = df["id"].astype("Int64")

print(df[numeric_cols].dtypes)
df[numeric_cols].describe(include="all").T


id                       Int64
precio_eur               int64
precio_anterior_eur    float64
descuento_pct          float64
superficie_m2            int64
dtype: object


,count,mean,std,min,25%,50%,75%,max
id,1080.0,540.5,311.91345,1.0,270.75,540.5,810.25,1080.0
precio_eur,1080.0,246821.639815,330320.65522,1200.0,76692.25,140000.0,269000.0,2900000.0
precio_anterior_eur,35.0,191800.0,172224.017933,58900.0,90900.0,125000.0,200000.0,695000.0
descuento_pct,35.0,13.771429,12.567666,2.0,5.0,10.0,18.0,50.0
superficie_m2,1080.0,6580.628704,12985.475222,1.0,1350.0,2934.0,6000.0,150000.0


## Eliminación de columna no requerida

Se elimina la variable `zona` como pediste.


In [41]:
if "zona" in df.columns:
    df = df.drop(columns=["zona"])

print("Columnas actuales:", df.columns.tolist())


Columnas actuales: ['id', 'anunciante', 'titulo', 'municipio', 'precio_eur', 'precio_anterior_eur', 'descuento_pct', 'superficie_m2', 'tipo_suelo', 'referencia_catastral', 'notas']


## Variable booleana de descuento

- Si `descuento_pct` tiene valor: `vendido_con_descuento = True`.
- Si `descuento_pct` está nulo: `descuento_pct` queda nulo y `vendido_con_descuento = False` (no descuento identificado).


In [42]:
df["vendido_con_descuento"] = df["descuento_pct"].notna()

df[["descuento_pct", "vendido_con_descuento"]].head(10)


,descuento_pct,vendido_con_descuento
0,NaN,False
1,NaN,False
2,NaN,False
3,NaN,False
4,NaN,False
5,NaN,False
6,NaN,False
7,6.0,True
8,NaN,False
9,NaN,False


## Limpieza de referencia catastral (string)

Se asegura tipo string y se convierte a nulo cuando no hay contenido real.


In [43]:
df["referencia_catastral"] = (
    df["referencia_catastral"]
    .astype("string")
    .str.strip()
)
df.loc[df["referencia_catastral"].isin(["", "nan", "None", "null"]), "referencia_catastral"] = pd.NA

print("Nulos en referencia_catastral:", int(df["referencia_catastral"].isna().sum()))


Nulos en referencia_catastral: 1028


## Normalización de `tipo_suelo` con apoyo de `notas`

Reglas aplicadas:
1. `Urbano (según texto)` -> `Urbano (solar)`.
2. Cuando `tipo_suelo` es vacío o `No especificado`, se infiere desde `notas`.
3. Si en `notas` aparecen a la vez urbano y rústico, prevalece `Urbano (solar)`.
4. Si no hay patrón claro, queda nulo.


In [44]:
def normalize_text(value: str) -> str:
    if pd.isna(value):
        return ""
    value = str(value).strip().lower()
    value = unicodedata.normalize("NFKD", value)
    return "".join(ch for ch in value if not unicodedata.combining(ch))

# Normalización base de tipo_suelo.
df["tipo_suelo"] = df["tipo_suelo"].astype("string").str.strip()
df.loc[df["tipo_suelo"].isin(["", "No especificado", "no especificado", "nan"]), "tipo_suelo"] = pd.NA
df["tipo_suelo"] = df["tipo_suelo"].replace({"Urbano (según texto)": "Urbano (solar)"})

mask_infer = df["tipo_suelo"].isna()

def infer_tipo_suelo_from_notas(notas: str):
    t = normalize_text(notas)

    has_urbano = bool(re.search(r"\burban[oa]s?\b|parcela urbana|suelo urbano", t))
    has_urbanizable = "urbanizable" in t
    has_industrial = "industrial" in t
    has_edificable = bool(re.search(r"edificable|edificabilidad", t))
    has_rustico = bool(re.search(r"rustic", t))

    if has_urbano and has_rustico:
        return "Urbano (solar)"
    if has_urbano or has_edificable:
        return "Urbano (solar)"
    if has_industrial:
        return "Industrial"
    if has_rustico:
        return "Rústico"
    if has_urbanizable:
        return "Urbanizable"
    return pd.NA

df.loc[mask_infer, "tipo_suelo"] = df.loc[mask_infer, "notas"].apply(infer_tipo_suelo_from_notas)

# Estandarizar variantes residuales.
df["tipo_suelo"] = df["tipo_suelo"].replace({
    "Urbano (no consolidado)": "Urbanizable"
})

print(df["tipo_suelo"].value_counts(dropna=False))


tipo_suelo
Urbano (solar)    558
Urbanizable       295
No urbanizable    182
<NA>               38
Industrial          5
Rústico             2
Name: count, dtype: Int64


## Variable booleana: urbano o urbanizable

`es_urbano_o_urbanizable` marca `True` para suelos urbanos/urbanizables y `False` para el resto (incluyendo industriales y rústicos).


In [45]:
df["es_urbano_o_urbanizable"] = df["tipo_suelo"].isin(["Urbano (solar)", "Urbanizable"])

df[["tipo_suelo", "es_urbano_o_urbanizable"]].head(10)


,tipo_suelo,es_urbano_o_urbanizable
0,<NA>,False
1,Urbano (solar),True
2,Urbano (solar),True
3,Urbano (solar),True
4,Urbano (solar),True
5,Urbano (solar),True
6,Urbanizable,True
7,Urbano (solar),True
8,No urbanizable,False
9,Urbano (solar),True


## Precio por metro cuadrado

Se calcula `precio_m2 = precio_eur / superficie_m2`, evitando divisiones por cero y superficies no válidas.


In [46]:
df.loc[df["superficie_m2"] <= 0, "superficie_m2"] = np.nan
df["precio_m2"] = df["precio_eur"] / df["superficie_m2"]

df[["precio_eur", "superficie_m2", "precio_m2"]].describe().T


,count,mean,std,min,25%,50%,75%,max
precio_eur,1080.0,246821.639815,330320.655220,1200.000000,76692.250000,140000.000000,269000.000000,2900000.0
superficie_m2,1080.0,6580.628704,12985.475222,1.000000,1350.000000,2934.000000,6000.000000,150000.0
precio_m2,1080.0,137.652114,1105.667214,0.610004,22.169811,56.322785,125.703735,36000.0


## Limpieza de título

Se elimina el prefijo `Terreno en ` al inicio del texto para estandarizar la variable `titulo`.


In [47]:
df["titulo"] = (
    df["titulo"]
    .astype("string")
    .str.replace(r"^Terreno en\s+", "", regex=True)
    .str.strip()
)

df["titulo"].head(10)


0    Barrio Llanilla, Alisal - Cazoña - San Román, ...
1                                      Oruña, Piélagos
2               Peñacastillo - Nuevamontaña, Santander
3    Calle de la Arnia, Soto de la Marina, Santa Cr...
4                      Calle Barrio San Roque, Polanco
5    Avenida Playa San Juan de la Canal, Soto de la...
6                    Barrio la Venera, Oruña, Piélagos
7                         Barrio Mar, 87 --81, Polanco
8                                  Sv-4713, 12, Miengo
9             Barrio la Picota, 38 a, Renedo, Piélagos
Name: titulo, dtype: string

## Control final y exportación opcional

Se revisa el resultado final y se deja una línea opcional para guardar el dataset limpio.


In [48]:
print("Shape final:", df.shape)
print("Nulos por columna:")
print(df.isna().sum().sort_values(ascending=False))

# Exportación opcional
# output_path = Path("..") / "data" / "1st_cleaning" / "terrenos_idealista_clean.csv"
# df.to_csv(output_path, sep=';', index=False)


Shape final: (1080, 14)
Nulos por columna:
precio_anterior_eur        1045
descuento_pct              1045
referencia_catastral       1028
notas                       807
anunciante                  436
tipo_suelo                   38
precio_eur                    0
municipio                     0
titulo                        0
id                            0
superficie_m2                 0
vendido_con_descuento         0
es_urbano_o_urbanizable       0
precio_m2                     0
dtype: int64
